<a href="https://colab.research.google.com/github/Seungkyu-Han/deep_learning/blob/main/ch5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [31]:
class MultiLayer:

  def __init__(
      self
  ):
    self.x = None
    self.y = None

  def forward(
      self,
      x: float,
      y: float,
  ) -> float:
    self.x = x
    self.y = y

    out = x * y

    return out

  def backward(
      self,
      value: float,
  ) -> tuple:
    dx = value * self.x
    dy = value * self.y

    return dx, dy

In [32]:
alpha_a = 100
num = 2
fee = 1.05

multi_alpha_layer = MultiLayer()
multi_fee_layer = MultiLayer()

stock_price = multi_alpha_layer.forward(alpha_a, num)
total_price = multi_fee_layer.forward(stock_price, fee)

print(total_price)

210.0


In [33]:
class PlusLayer:

  def forward(
      self,
      x: float,
      y: float,
  ) -> float:
    out = x + y
    return out

  def backward(
      self,
      value: float,
  ) -> tuple:
    dx = value
    dy = value

    return dx, dy

In [34]:
class ReluNode:

  def __init__(
      self,
  ):
    self.mask = None

  def forward(
      self,
      x,
  ):
    self.mask = (x <= 0)
    out = x.copy()
    out[self.mask] = 0

    return out

  def backward(
      self,
      value,
  ):
    value[self.mask] = 0
    dx = value

    return dx

In [35]:
import numpy as np

class SigmoidNode:

  def __init__(
      self,
  ):
    self.out = None

  def forward(
      self,
      x,
  ):
    out = 1 / (1 + np.exp(-x))
    self.out = out

    return out

  def backward(
      self,
      value,
  ):
    dx = value * (1.0 - self.out) * self.out

    return dx

In [36]:
class AffineNode:

  def __init__(
      self,
      W,
      b,
  ):
    self.W = W
    self.b = b
    self.x = None
    self.dW = None
    self.db = None

  def forward(
      self,
      x,
  ):
    self.x = x
    out = np.dot(x, self.W) + self.b

    return out

  def backward(
      self,
      value,
  ):
    dx = np.dot(value, self.W.T)
    self.dW = np.dot(self.x.T, value)
    self.db = np.sum(value, axis=0)

    return dx


In [37]:
def cross_entropy_error(y, t):
  delta = 1e-8
  return - np.sum(t * np.log(y + delta))

In [38]:
def softmax(x):
    # 2차원 배치 데이터인 경우와 1차원 데이터인 경우를 모두 대응
    if x.ndim == 2:
        x = x.T
        # 각 데이터(열)별 최댓값을 찾아서 빼줍니다. (오버플로우 방지)
        x = x - np.max(x, axis=0)
        y = np.exp(x) / np.sum(np.exp(x), axis=0)
        return y.T

    # 1차원 데이터인 경우
    c = np.max(x)
    exp_a = np.exp(x - c)  # 최댓값을 빼주어 안전하게 만듦 (0 이하의 값으로 변환됨)
    sum_exp_a = np.sum(exp_a)
    y = exp_a / sum_exp_a

    return y

In [39]:
class SoftmaxWithLossNode:

  def __init__(
      self,
  ):
    self.loss = None
    self.y = None
    self.x = None

  def forward(
      self,
      x,
      t,
  ):
    self.t = t
    self.y = softmax(x)
    self.loss = cross_entropy_error(self.y, self.t)

    return self.loss

  def backward(
      self,
      value = 0,
  ):
    batch_size = self.t.shape[0]
    dx = (self.y - self.t) / batch_size

    return dx

In [40]:
def numerical_gradient(f, x):
    h = 1e-4  # 0.0001
    grad = np.zeros_like(x)  # x와 형상이 같은 배열을 생성

    # 다차원 배열의 인덱스를 하나씩 순회할 수 있게 해주는 넘파이 반복자
    it = np.nditer(x, flags=['multi_index'], op_flags=['readwrite'])

    while not it.finished:
        idx = it.multi_index  # 현재의 다차원 인덱스 추출 (예: (0, 0), (0, 1)...)
        tmp_val = x[idx]

        # f(x + h) 계산
        x[idx] = float(tmp_val) + h
        fxh1 = f(x)

        # f(x - h) 계산
        x[idx] = float(tmp_val) - h
        fxh2 = f(x)

        # 중앙 차분으로 미분값 계산
        grad[idx] = (fxh1 - fxh2) / (2 * h)

        x[idx] = tmp_val  # 값 원상 복구
        it.iternext()     # 다음 원소로 이동

    return grad

In [41]:
import sys, os

import numpy as np

from collections import OrderedDict

class TwoLayerNetwork:

  def __init__(
      self,
      input_size,
      hidden_size,
      output_size,
      weight_init_std=0.01,
  ):
    self.params = {}

    # [수정] weight2의 크기를 (hidden_size, output_size)로 정상화
    self.params['weight1'] = weight_init_std * np.random.randn(input_size, hidden_size)
    self.params['weight2'] = weight_init_std * np.random.randn(hidden_size, output_size)
    self.params['b1'] = np.zeros(hidden_size)
    self.params['b2'] = np.zeros(output_size)

    self.layers = OrderedDict()
    self.layers['Affine1'] = AffineNode(
        self.params['weight1'],
        self.params['b1'],
        )
    self.layers['Relu1'] = ReluNode()
    self.layers['Affine2'] = AffineNode(
        self.params['weight2'],
        self.params['b2'],
        )

    self.lastlayer = SoftmaxWithLossNode()

  def predict(
      self,
      x,
  ):
    for layer in self.layers.values():
      x = layer.forward(x)
    return x

  def loss(
      self,
      x,
      t,
  ):
    y = self.predict(x)
    # [수정] lastLayer -> lastlayer 대소문자 고침
    return self.lastlayer.forward(y, t)

  def accuracy(
      self,
      x,
      t,
  ):
    y = self.predict(x)
    y = np.argmax(y, axis=1)
    if t.ndim != 1:
        t = np.argmax(t, axis=1)

    accuracy = np.sum(y == t) / float(x.shape[0])
    return accuracy

  def numerical_gradient(
      self,
      x,
      t,
  ):
    loss_w = lambda W: self.loss(x, t)

    grads = {}
    grads['weight1'] = numerical_gradient(loss_w, self.params['weight1'])
    grads['b1'] = numerical_gradient(loss_w, self.params['b1'])
    grads['weight2'] = numerical_gradient(loss_w, self.params['weight2'])
    grads['b2'] = numerical_gradient(loss_w, self.params['b2'])

    return grads

  def gradient(
      self,
      x,
      t,
  ):
    # 1. 순전파를 먼저 쭉 흘려보내서 각 Node에 forward 값(x, y 등)을 기억시킴
    self.loss(x, t)

    # 2. 역전파 시작 (마지막 레이어부터 역순으로)
    value = 1
    value = self.lastlayer.backward(value)

    layers = list(self.layers.values())
    layers.reverse()

    for layer in layers:
      value = layer.backward(value)

    # 3. [수정] 각 AffineNode가 역전파 때 계산해서 보관해둔 dW, db를 수집
    grads = {}
    grads['weight1'] = self.layers['Affine1'].dW
    grads['b1'] = self.layers['Affine1'].db
    grads['weight2'] = self.layers['Affine2'].dW
    grads['b2'] = self.layers['Affine2'].db

    return grads

In [42]:
# coding: utf-8
import os
import gzip
import pickle
import urllib.request
import numpy as np

# [수정] Yann LeCun 서버 불안정 문제를 해결하기 위해 미러링(Cloud) 서버 주소로 변경
url_base = 'https://ossci-datasets.s3.amazonaws.com/mnist/'
key_file = {
    'train_img':'train-images-idx3-ubyte.gz',
    'train_label':'train-labels-idx1-ubyte.gz',
    'test_img':'t10k-images-idx3-ubyte.gz',
    'test_label':'t10k-labels-idx1-ubyte.gz'
}

# [수정] 코랩 환경에는 __file__ 변수가 없으므로 기본 /content 경로로 설정합니다.
dataset_dir = "/content"
save_file = os.path.join(dataset_dir, "mnist.pkl")

train_num = 60000
test_num = 10000
img_dim = (1, 28, 28)
img_size = 784


def _download(file_name):
    file_path = os.path.join(dataset_dir, file_name)

    if os.path.exists(file_path):
        return

    print("Downloading " + file_name + " ... ")
    # 코랩 환경에서 다운로드 차단을 막기 위해 User-Agent 헤더 추가
    headers = {'User-Agent': 'Mozilla/5.0'}
    req = urllib.request.Request(url_base + file_name, headers=headers)

    with urllib.request.urlopen(req) as response, open(file_path, 'wb') as out_file:
        out_file.write(response.read())
    print("Done")

def download_mnist():
    for v in key_file.values():
       _download(v)

def _load_label(file_name):
    file_path = os.path.join(dataset_dir, file_name)

    print("Converting " + file_name + " to NumPy Array ...")
    with gzip.open(file_path, 'rb') as f:
            labels = np.frombuffer(f.read(), np.uint8, offset=8)
    print("Done")

    return labels

def _load_img(file_name):
    file_path = os.path.join(dataset_dir, file_name)

    print("Converting " + file_name + " to NumPy Array ...")
    with gzip.open(file_path, 'rb') as f:
            data = np.frombuffer(f.read(), np.uint8, offset=16)
    data = data.reshape(-1, img_size)
    print("Done")

    return data

def _convert_numpy():
    dataset = {}
    dataset['train_img'] =  _load_img(key_file['train_img'])
    dataset['train_label'] = _load_label(key_file['train_label'])
    dataset['test_img'] = _load_img(key_file['test_img'])
    dataset['test_label'] = _load_label(key_file['test_label'])

    return dataset

def init_mnist():
    download_mnist()
    dataset = _convert_numpy()
    print("Creating pickle file ...")
    with open(save_file, 'wb') as f:
        pickle.dump(dataset, f, -1)
    print("Done!")

def _change_one_hot_label(X):
    T = np.zeros((X.size, 10))
    for idx, row in enumerate(T):
        row[X[idx]] = 1

    return T


def load_mnist(normalize=True, flatten=True, one_hot_label=False):
    """MNIST 데이터셋 읽기

    Parameters
    ----------
    normalize : 이미지의 픽셀 값을 0.0~1.0 사이의 값으로 정규화할지 정한다.
    one_hot_label :
        one_hot_label이 True면、레이블을 원-핫(one-hot) 배열로 돌려준다.
    flatten : 입력 이미지를 1차원 배열로 만들지를 정한다.

    Returns
    -------
    (훈련 이미지, 훈련 레이블), (시험 이미지, 시험 레이블)
    """
    if not os.path.exists(save_file):
        init_mnist()

    with open(save_file, 'rb') as f:
        dataset = pickle.load(f)

    if normalize:
        for key in ('train_img', 'test_img'):
            dataset[key] = dataset[key].astype(np.float32)
            dataset[key] /= 255.0

    if one_hot_label:
        dataset['train_label'] = _change_one_hot_label(dataset['train_label'])
        dataset['test_label'] = _change_one_hot_label(dataset['test_label'])

    if not flatten:
         for key in ('train_img', 'test_img'):
            dataset[key] = dataset[key].reshape(-1, 1, 28, 28)

    return (dataset['train_img'], dataset['train_label']), (dataset['test_img'], dataset['test_label'])


# ==========================================
# 실행 테스트 코드
# ==========================================
# 이제 아래 함수를 실행하면 코랩 내부로 다운로드부터 포맷팅까지 한 번에 끝납니다!
(x_train, t_train), (x_test, t_test) = load_mnist(normalize=True, flatten=True, one_hot_label=True)

print("\n데이터 다운로드 및 로드 성공!")
print("x_train 구조:", x_train.shape)
print("t_train 구조:", t_train.shape)


데이터 다운로드 및 로드 성공!
x_train 구조: (60000, 784)
t_train 구조: (60000, 10)


In [43]:
(x_train, t_train), (x_test, t_test) = load_mnist(
    normalize=True,
    one_hot_label=True,
)

network = TwoLayerNetwork(
    input_size=784, hidden_size = 10, output_size = 10
)

x_batch = x_train[:3]
t_batch = t_train[:3]

grad_numerical = network.numerical_gradient(
    x_batch, t_batch,
)
grad_backprop = network.gradient(
    x_batch, t_batch,
)

for key in grad_numerical.keys():
  diff = np.average(np.abs(grad_backprop[key] - grad_numerical[key]))
  print(key + ": " + str(diff))

weight1: 0.0003632163028221342
b1: 0.0019688735161299104
weight2: 0.006658534736679539
b2: 0.28001586376604626


In [44]:
(x_train, t_train), (x_test, t_test) = load_mnist(
    normalize=True,
    one_hot_label=True,
)

network = TwoLayerNetwork(
    input_size=784, hidden_size = 10, output_size = 10
)

iters_num = 10000
train_size, t_train_size = x_train.shape[0], t_train.shape[0]
x_test_size, t_test_size = x_test.shape[0], t_test.shape[0]
print(train_size, t_train_size)
print(x_test_size, t_test_size)

60000 60000
10000 10000


In [45]:
batch_size = 100
learning_rate = 0.1

train_loss_list = []
train_acc_list = []
test_acc_list = []

iter_per_epoch = max(train_size // batch_size, 1)

for i in range(iters_num):

  batch_mask = np.random.choice(train_size, batch_size)

  x_batch = x_train[batch_mask]
  t_batch = t_train[batch_mask]

  grad = network.gradient(x_batch, t_batch)

  for key in ('weight1', 'b1', 'weight2', 'b2'):
    network.params[key] -= learning_rate * grad[key]

  loss = network.loss(x_batch, t_batch)

  train_loss_list.append(loss)

  if i % iter_per_epoch == 0:
    train_acc = network.accuracy(x_train, t_train)
    test_acc = network.accuracy(x_test, t_test)
    train_acc_list.append(train_acc)
    test_acc_list.append(test_acc)
    print(train_acc, test_acc)



0.12118333333333334 0.1203
0.88885 0.8914
0.9104833333333333 0.9113
0.91425 0.9151
0.9216333333333333 0.919
0.9196 0.9181
0.9214333333333333 0.9218
0.9314833333333333 0.9282
0.9347833333333333 0.9334
0.9354333333333333 0.9331
0.9374833333333333 0.9352
0.9383166666666667 0.9358
0.93915 0.9336
0.9405 0.9357
0.9411833333333334 0.9377
0.9435166666666667 0.9379
0.94245 0.9379
